##  Setup and Test Scenarios

In [1]:
import os
import json
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
model = "gpt-5-mini"

DEPARTMENTS = {
    "rental-contracts":     "Rental Contracts — lease questions, renewals, amendments",
    "terminations-moveout": "Terminations & Move-out — cancellations, move-out dates, deposit returns",
    "tenant-complaints":    "Tenant Complaints — noise, neighbour disputes, general grievances",
    "energy-heating":       "Energy & Heating — heating failures, hot water, utility billing",
    "repairs-maintenance":  "Repairs & Maintenance — broken fixtures, structural damage, general repairs",
}

TEST_SCENARIOS = [
    {
        "id": "TC-001",
        "description": "Clear heating issue, cooperative tenant",
        "persona": (
            "You are Lisa Müller, a tenant at Berliner Str. 12, Berlin. "
            "Your heating stopped working 2 days ago. You are polite but worried because it's cold. "
            "Give your name and address when asked. Keep answers short and factual."
        ),
        "expected_department": "energy-heating",
        "max_turns": 5,
        "min_friendliness": 3,
    },
    {
        "id": "TC-002",
        "description": "Lease termination, calm and direct",
        "persona": (
            "You are Thomas Bauer, a tenant at Hauptstr. 7, Hamburg. "
            "You want to end your lease. You are calm and businesslike. "
            "Give your name and address when asked."
        ),
        "expected_department": "terminations-moveout",
        "max_turns": 5,
        "min_friendliness": 2,
    },
    {
        "id": "TC-003",
        "description": "Noise complaint, frustrated tone",
        "persona": (
            "You are Emre Yilmaz, a tenant at Gartenweg 3, Munich. "
            "Your upstairs neighbour makes noise every night and you haven't slept in a week. "
            "You are frustrated and slightly impatient. Give your details when asked."
        ),
        "expected_department": "tenant-complaints",
        "max_turns": 6,
        "min_friendliness": 3,
    },
    {
        "id": "TC-004",
        "description": "Auth failure — wrong address",
        "persona": (
            "You are Sandra Koch. You want to report a broken window. "
            "When asked for your address, say 'Musterstr. 99, Berlin' — this is not a real tenant address. "
            "Do not correct it even if the agent says it can't be verified."
        ),
        "expected_department": "auth-failed",   # special value: we expect the call to end here
        "max_turns": 3,
        "min_friendliness": 3,
    },
    {
        "id": "TC-005",
        "description": "Ambiguous issue — agent must ask, not guess",
        "persona": (
            "You are Klaus Werner, a tenant at Friedrichstr. 45, Berlin. "
            "Your opening message is just: 'I have a problem with my apartment.' "
            "Only give more details if the agent asks a direct question. Be cooperative."
        ),
        "expected_department": None,            # any correct department is acceptable — we test that the agent asks first
        "max_turns": 6,
        "min_friendliness": 3,
    },
    {
        "id": "TC-006",
        "description": "Tricky phrasing — sounds like terminations but is energy",
        "persona": (
            "You are Yuki Tanaka, a tenant at Rosenweg 8, Frankfurt. "
            "You want to cancel your energy supply contract, not your apartment lease. "
            "Say: 'I want to cancel my energy contract.' Give your details when asked."
        ),
        "expected_department": "energy-heating",
        "max_turns": 5,
        "min_friendliness": 2,
    },
]

## The Routing Agent (Standalone Version)

In [2]:
from pydantic import BaseModel

class AuthResult(BaseModel):
    verified: bool
    customer_name: str
    reason: str

class RoutingDecision(BaseModel):
    department: str
    routing_reason: str
    issue_summary: str
    confidence: str  # "low" / "medium" / "high"

ROUTING_SYSTEM_PROMPT = (
    "You are a routing agent at Stadtquartier Wohnservice, a residential property management company.\n"
    "Based on the conversation transcript, decide which department should handle this tenant's issue.\n\n"
    "Available departments:\n"
    + "\n".join(f"- {k}: {v}" for k, v in DEPARTMENTS.items())
    + "\n\nChoose exactly one department key from the list above. "
    "In routing_reason, explain specifically why this department is the right fit."
)

def route_conversation(transcript: list[dict]) -> RoutingDecision | None:
    """Takes a list of {"role": "tenant"/"agent", "text": "..."} dicts, returns routing decision."""
    transcript_text = "\n".join(f"{t['role'].upper()}: {t['text']}" for t in transcript)
    try:
        response = client.beta.chat.completions.parse(
            model=model,
            messages=[
                {"role": "system", "content": ROUTING_SYSTEM_PROMPT},
                {"role": "user", "content": f"Conversation:\n{transcript_text}"}
            ],
            response_format=RoutingDecision
        )
        return response.choices[0].message.parsed
    except Exception:
        return None

def check_auth(name: str, address: str) -> AuthResult:
    response = client.beta.chat.completions.parse(
        model=model,
        messages=[
            {"role": "system", "content": (
                "You simulate a tenant database check. "
                "If the address has a recognisable street name and number, mark verified. "
                "Clearly fictional or incomplete addresses (like 'Musterstr. 99') are not verified."
            )},
            {"role": "user", "content": f"Name: {name}\nAddress: {address}"}
        ],
        response_format=AuthResult
    )
    return response.choices[0].message.parsed

## Test Agent

In [3]:
def test_agent_message(persona: str, conversation_history: list[dict]) -> str:
    """Given a persona and the conversation so far, generate the next customer message."""
    messages = [
        {"role": "system", "content": (
            f"{persona}\n\n"
            "You are participating in a customer support call. "
            "Respond naturally as this person would. "
            "Keep each response short — 1–3 sentences maximum. "
            "Do not break character."
        )}
    ]
    # Add conversation history: agent messages become 'user' from the test agent's perspective
    for turn in conversation_history:
        role = "assistant" if turn["role"] == "tenant" else "user"
        messages.append({"role": role, "content": turn["text"]})

    messages.append({"role": "user", "content": "What do you say next?"})

    response = client.chat.completions.create(model=model, messages=messages)
    return response.choices[0].message.content.strip()

## Conversation Runner

In [4]:
AGENT_GREETING = (
    "Thank you for calling Stadtquartier Wohnservice. "
    "My name is Alex, how can I help you today?"
)

def run_test_conversation(scenario: dict) -> dict:
    """Runs one full test conversation. Returns transcript + routing result."""
    conversation = []
    routing_result = None
    auth_failed = False

    # Agent opens
    conversation.append({"role": "agent", "text": AGENT_GREETING})

    for turn_num in range(scenario["max_turns"] + 2):  # small buffer
        # Test agent responds as customer
        customer_msg = test_agent_message(scenario["persona"], conversation)
        conversation.append({"role": "tenant", "text": customer_msg})

        # Simple auth extraction (look for name + address pattern)
        # In a real system this would be the step-auth span from session 2
        tenant_messages = " ".join(t["text"] for t in conversation if t["role"] == "tenant")
        if "Musterstr" in tenant_messages or "99" in tenant_messages:
            auth_failed = True
            conversation.append({
                "role": "agent",
                "text": "I'm sorry, I wasn't able to verify your details with that address. Please contact us directly."
            })
            break

        # After enough context (3+ turns), attempt routing
        if len(conversation) >= 5:
            routing_result = route_conversation(conversation)
            if routing_result and routing_result.confidence in ("medium", "high"):
                agent_reply = (
                    f"Thank you. I'm routing you to our {DEPARTMENTS.get(routing_result.department, routing_result.department)} team. "
                    f"{routing_result.routing_reason}"
                )
                conversation.append({"role": "agent", "text": agent_reply})
                break
        else:
            # Agent asks a clarifying question
            response = client.chat.completions.create(
                model=model,
                messages=[
                    {"role": "system", "content": "You are a friendly receptionist. Ask for the tenant's name, address, or clarify their issue. Keep it to one short question."},
                    {"role": "user", "content": "\n".join(f"{t['role']}: {t['text']}" for t in conversation)}
                ]
            )
            agent_reply = response.choices[0].message.content.strip()
            conversation.append({"role": "agent", "text": agent_reply})

    return {
        "scenario_id": scenario["id"],
        "transcript": conversation,
        "routing": routing_result,
        "auth_failed": auth_failed,
    }

## The Evaluator

In [5]:
FRIENDLINESS_RUBRIC = """
Score the AGENT's tone on a scale of 1 to 5:
  1 = cold or dismissive
  3 = professional but flat
  5 = warm and empathetic
Return JSON: {"score": <1-5>, "reason": "<one sentence>"}
"""

def evaluate_result(scenario: dict, result: dict) -> dict:
    transcript = result["transcript"]
    routing = result["routing"]
    auth_failed = result["auth_failed"]

    # Routing match
    if scenario["expected_department"] == "auth-failed":
        routing_correct = auth_failed
    elif scenario["expected_department"] is None:
        routing_correct = True  # TC-005: any routing is fine, we check separately
    else:
        routing_correct = (routing is not None and routing.department == scenario["expected_department"])

    # Turn count
    turn_count = len([t for t in transcript if t["role"] == "tenant"])
    within_budget = turn_count <= scenario["max_turns"]

    # Friendliness (LLM-as-Judge)
    convo_text = "\n".join(f"{t['role'].upper()}: {t['text']}" for t in transcript)
    judge_response = client.chat.completions.create(
        model="gpt-5-mini",
        response_format={"type": "json_object"},
        messages=[
            {"role": "system", "content": FRIENDLINESS_RUBRIC},
            {"role": "user", "content": convo_text}
        ]
    )
    friendliness_result = json.loads(judge_response.choices[0].message.content)
    friendliness_score = friendliness_result["score"]

    passed = routing_correct and within_budget and friendliness_score >= scenario["min_friendliness"]

    return {
        "id": scenario["id"],
        "description": scenario["description"],
        "expected": scenario["expected_department"] or "any",
        "actual": routing.department if routing else ("auth-failed" if auth_failed else "no-routing"),
        "routing_correct": routing_correct,
        "turns": turn_count,
        "within_budget": within_budget,
        "friendliness": friendliness_score,
        "routing_reason": routing.routing_reason if routing else "—",
        "passed": passed,
    }

## Test Runner And Reporter

In [9]:
def run_suite(scenarios: list) -> list:
    results = []
    for scenario in scenarios:
        print(f"Running {scenario['id']}: {scenario['description']}...")
        conversation_result = run_test_conversation(scenario)
        eval_result = evaluate_result(scenario, conversation_result)
        results.append(eval_result)
        status = "✅ PASS" if eval_result["passed"] else "❌ FAIL"
        print(f"  {status} — routing: {eval_result['actual']} | turns: {eval_result['turns']} | friendliness: {eval_result['friendliness']}/5")
    return results

def print_report(results: list):
    print("\n" + "=" * 75)
    print(f"{'ID':<8} {'Expected':<22} {'Actual':<22} {'Turns':>5} {'Friend':>7} {'Result':>6}")
    print("-" * 75)
    for r in results:
        status = "PASS" if r["passed"] else "FAIL"
        print(
            f"{r['id']:<8} {r['expected']:<22} {r['actual']:<22} "
            f"{r['turns']:>5} {r['friendliness']:>7} {status:>6}"
        )

    passed = sum(1 for r in results if r["passed"])
    total = len(results)
    routing_acc = sum(1 for r in results if r["routing_correct"]) / total
    avg_turns = sum(r["turns"] for r in results) / total
    avg_friendliness = sum(r["friendliness"] for r in results) / total

    print("=" * 75)
    print(f"SUMMARY: {passed}/{total} passed | routing accuracy: {routing_acc:.0%} | avg turns: {avg_turns:.1f} | avg friendliness: {avg_friendliness:.1f}/5")

if __name__ == "__main__":
    results = run_suite(TEST_SCENARIOS)
    print_report(results)

Running TC-001: Clear heating issue, cooperative tenant...
  ❌ FAIL — routing: energy-heating | turns: 7 | friendliness: 4/5
Running TC-002: Lease termination, calm and direct...
  ❌ FAIL — routing: terminations-moveout | turns: 7 | friendliness: 4/5
Running TC-003: Noise complaint, frustrated tone...
  ❌ FAIL — routing: tenant-complaints | turns: 8 | friendliness: 4/5
Running TC-004: Auth failure — wrong address...
  ✅ PASS — routing: auth-failed | turns: 1 | friendliness: 3/5
Running TC-005: Ambiguous issue — agent must ask, not guess...
  ✅ PASS — routing: repairs-maintenance | turns: 5 | friendliness: 3/5
Running TC-006: Tricky phrasing — sounds like terminations but is energy...
  ❌ FAIL — routing: energy-heating | turns: 7 | friendliness: 4/5

ID       Expected               Actual                 Turns  Friend Result
---------------------------------------------------------------------------
TC-001   energy-heating         energy-heating             7       4   FAIL
TC-002   ter

##  Break Something — Watch the Suite Go Red

In [7]:
# Break the routing prompt:
ROUTING_SYSTEM_PROMPT = (
    "You are a routing agent. Route the tenant to a department. "
    "Available departments:\n"
    + "\n".join(f"- {k}" for k in DEPARTMENTS.keys())
    # ponytail: removed department descriptions and routing_reason instruction intentionally
)

## fIX iT

In [8]:
ROUTING_SYSTEM_PROMPT = (
    "You are a routing agent at Stadtquartier Wohnservice, a residential property management company.\n"
    "Based on the conversation transcript, decide which department should handle this tenant's issue.\n\n"
    "Available departments:\n"
    + "\n".join(f"- {k}: {v}" for k, v in DEPARTMENTS.items())
    + "\n\nChoose exactly one department key. "
    "In routing_reason, explain specifically why this department is the right fit."
)